# Website Figure Tutorial

This notebook builds the astronomy-focused website background figure using the generalized `oviz` API: `Layer`, `LayerCollection`, `Scene3D`, and Three.js profile helpers.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import FileLink, IFrame, display

from oviz import Layer, LayerCollection, Scene3D, build_threejs_profile, merge_threejs_profile


In [ ]:
HOME = Path.home()

galaxy_image = True

cluster_catalog_path = HOME / "Desktop" / "astro_research" / "supernovae_map_work" / "clusters" / "vels_output" / "2026-02-04" / "cluster_velocities_jan2026.csv"
chronos_age_path = HOME / "Downloads" / "hunt_sample_chronos_ages_multiprocessing_feb_2026.csv"
cluster_family_catalog_path = HOME / "Downloads" / "cluster_sample_data.csv"
edenhofer_fits_path = HOME / "Downloads" / "mean_and_std_xyz-2.fits"
galactic_plane_image_path = HOME / "Downloads" / "Top-down_view_of_the_Milky_Way.jpg"

output_html = Path("/tmp/website_figure_tutorial.html")

required_paths = {
    "cluster_catalog": cluster_catalog_path,
    "chronos_ages": chronos_age_path,
    "cluster_family_catalog": cluster_family_catalog_path,
    "edenhofer_fits": edenhofer_fits_path,
}
if galaxy_image:
    required_paths["galaxy_image"] = galactic_plane_image_path

missing = []
for label, path in required_paths.items():
    exists = path.exists()
    print(f"{label:24s} {path} {'OK' if exists else 'MISSING'}")
    if not exists:
        missing.append(label)

if missing:
    raise FileNotFoundError("Missing required local inputs: " + ", ".join(missing))

output_html


In [ ]:
df_hunt_full = pd.read_csv(cluster_catalog_path)
ages_chronos = pd.read_csv(chronos_age_path)
cluster_family_catalog = pd.read_csv(cluster_family_catalog_path)

df_hunt_full = pd.merge(
    df_hunt_full,
    ages_chronos[["name", "age_chronos_lo", "age_chronos_mode", "age_chronos_hi"]],
    on="name",
    how="left",
)
df_hunt_full["age_myr"] = df_hunt_full["age_chronos_mode"]

df_hunt_full = df_hunt_full.drop(columns=["x", "y", "z", "U", "V", "W", "U_err", "V_err", "W_err"])
df_hunt_full = df_hunt_full.rename(
    columns={
        "U_2026": "U",
        "V_2026": "V",
        "W_2026": "W",
        "U_err_2026": "U_err",
        "V_err_2026": "V_err",
        "W_err_2026": "W_err",
        "x_new": "x",
        "y_new": "y",
        "z_new": "z",
    }
)

df_hunt_good = df_hunt_full.loc[
    (df_hunt_full["U_err"] < 10)
    & (df_hunt_full["V_err"] < 10)
    & (df_hunt_full["W_err"] < 10)
    & (df_hunt_full["U"].notnull())
    & (df_hunt_full["V"].notnull())
    & (df_hunt_full["W"].notnull())
    & (df_hunt_full["age_myr"].notnull())
    & (df_hunt_full["x"].between(-2000, 2000))
    & (df_hunt_full["y"].between(-2000, 2000))
    & (df_hunt_full["z"].between(-300, 300))
    & (df_hunt_full["n_rvs_2026"] >= 3)
    & (df_hunt_full["class_50"] > 0.5)
    & (df_hunt_full["cst"] > 5)
].copy()

df_hunt_60 = df_hunt_good.loc[df_hunt_good["age_myr"] < 60].copy()

alpha_per_family = df_hunt_60.loc[
    df_hunt_60["name"].isin(cluster_family_catalog.loc[cluster_family_catalog["family"] == "alphaPer", "name"])
].copy()
cr135_family = df_hunt_60.loc[
    df_hunt_60["name"].isin(cluster_family_catalog.loc[cluster_family_catalog["family"] == "cr135", "name"])
].copy()
m6_family = df_hunt_60.loc[
    df_hunt_60["name"].isin(cluster_family_catalog.loc[cluster_family_catalog["family"] == "m6", "name"])
].copy()

sun = pd.DataFrame(
    {
        "name": ["Sun"],
        "age_myr": [8000.0],
        "U": [0.0],
        "V": [0.0],
        "W": [0.0],
        "x": [0.0],
        "y": [0.0],
        "z": [27.0],
        "n_stars": [1],
    }
)

print(f"Total good clusters: {len(df_hunt_good):,}")
print(f"Clusters < 60 Myr: {len(df_hunt_60):,}")
print(f"Alpha Persei Family: {len(alpha_per_family):,}")
print(f"Cr 135 Family: {len(cr135_family):,}")
print(f"M6 Family: {len(m6_family):,}")


In [ ]:
cluster_layer = Layer.from_dataframe(
    df_hunt_60,
    layer_name="Clusters (< 60 Myr)",
    min_size=0.0,
    max_size=7.0,
    color="#5fd6ff",
    opacity=0.72,
    marker_style="circle",
    show_tracks=False,
    size_by_n_stars=False,
)

sun_layer = Layer.from_dataframe(
    sun,
    layer_name="Sun",
    min_size=5.0,
    max_size=5.0,
    color="yellow",
    opacity=1.0,
    marker_style="circle",
    show_tracks=False,
    size_by_n_stars=False,
)

alpha_per_layer = Layer.from_dataframe(alpha_per_family, layer_name="Alpha Persei Family", min_size=0.0, max_size=7.0, color="violet", opacity=1.0, marker_style="circle")
cr135_layer = Layer.from_dataframe(cr135_family, layer_name="Cr 135 Family", min_size=0.0, max_size=7.0, color="orange", opacity=1.0, marker_style="circle")
m6_layer = Layer.from_dataframe(m6_family, layer_name="M6 Family", min_size=0.0, max_size=7.0, color="cyan", opacity=1.0, marker_style="circle")

layers = LayerCollection([sun_layer, cluster_layer, alpha_per_layer, cr135_layer, m6_layer])
layer_groupings = {
    "Clusters": ["Sun", "Clusters (< 60 Myr)"],
    "Swiggum et al. (2024)": ["Sun", "Alpha Persei Family", "Cr 135 Family", "M6 Family"],
}

layers.get_all_layers()


In [ ]:
greys_cmap = plt.get_cmap("Greys")

edenhofer_volume = {
    "name": "Edenhofer+2024 Dust",
    "path": str(edenhofer_fits_path),
    "hdu": "MEAN",
    "max_resolution": 100,
    "opacity": 1.0,
    "samples": 24,
    "alpha_coef": 200,
    "vmin": 0.0,
    "vmax": 0.0385,
    "colormap": greys_cmap,
    "only_at_t0": True,
}

threejs_initial_state = merge_threejs_profile(
    build_threejs_profile(
        "website_background",
        galaxy_image=galaxy_image,
        galaxy_image_path=str(galactic_plane_image_path) if galaxy_image else None,
    ),
    {
        "current_group": "Clusters",
        "active_volume_key": "volume-0",
        "legend_state": {"volume-0": True},
        "volume_state_by_key": {"volume-0": {"visible": True}},
    },
)

actions = [
    {
        "key": "swiggum-2024",
        "label": "Swiggum et al. (2024), Nature",
        "description": "Show the Alpha Persei, Cr 135, and M6 families.",
        "steps": [
            {"type": "legend_group", "group": "Swiggum et al. (2024)", "duration_ms": 420},
            {
                "type": "camera",
                "target": {"kind": "group", "name": "Swiggum et al. (2024)"},
                "anchor_target": {"kind": "point", "x": 0.0, "y": 0.0, "z": 0.0},
                "duration_ms": 1200,
                "camera": {
                    "position": {"x": 360.0, "y": -3400.0, "z": 2440.0},
                    "view_offset": {"x": 0.22, "y": 0.0},
                },
            },
            {
                "type": "time",
                "start": "after_previous",
                "delay_ms": 120,
                "direction": "backward",
                "interval_ms": 80,
                "stop_after_frames": 30,
            },
            {
                "type": "camera",
                "target": {"kind": "group", "name": "Swiggum et al. (2024)"},
                "anchor_target": {"kind": "point", "x": 0.0, "y": 0.0, "z": 0.0},
                "duration_ms": 0,
                "camera": {
                    "position": {"x": 360.0, "y": -3400.0, "z": 2440.0},
                    "view_offset": {"x": 0.22, "y": 0.0},
                },
                "orbit": {"enabled": True, "speed_multiplier": 1.0, "direction": 1, "persist": True},
            },
        ],
    }
]

threejs_initial_state


In [ ]:
time_int = np.round(np.arange(0, -61, -1), 1)

scene = Scene3D(
    data_collection=layers,
    xyz_widths=(2000, 2000, 400),
    figure_theme="dark",
    trace_grouping_dict=layer_groupings,
)

fig3d = scene.make_plot(
    time=time_int,
    show=False,
    save_name=str(output_html),
    focus_group=None,
    fade_in_time=8,
    fade_in_and_out=False,
    show_age_kde_inset=False,
    include_spiral_arms=False,
    galactic_mode=True,
    show_galactic_guides=False,
    camera_zoom_factor=5,
    show_gc_line=True,
    show_milky_way_model=False,
    renderer="threejs",
    enable_sky_panel=False,
    volumes=[edenhofer_volume],
    threejs_initial_state=threejs_initial_state,
    actions=actions,
)

fig3d.write_html(output_html)

output_html.exists(), output_html


In [ ]:
display(FileLink(output_html))
IFrame(src=output_html.as_uri(), width="100%", height=720)
